# Silver Layer — Hospital Data Inspector

Queries `s3://silver/hospitals.parquet` directly from MinIO using DuckDB.

In [1]:
import os

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

path = "s3://silver/hospitals.parquet"

con = duckdb.connect()
for ext in ("spatial", "httpfs"):
    con.install_extension(ext)
    con.load_extension(ext)

endpoint = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
con.sql(f"""
    CREATE SECRET minio (
        TYPE s3,
        KEY_ID '{os.environ["MINIO_ACCESSKEY"]}',
        SECRET '{os.environ["MINIO_SECRETKEY"]}',
        ENDPOINT '{endpoint}',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

%sql con --alias duckdb

## Schema

In [2]:
%%sql
SELECT * FROM (DESCRIBE SELECT * FROM '{{path}}')

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,geometry,GEOMETRY('OGC:CRS84'),YES,None,None,None
3,geometry_3035,GEOMETRY('EPSG:3035'),YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,city,VARCHAR,YES,None,None,None
6,street,VARCHAR,YES,None,None,None
7,postcode,VARCHAR,YES,None,None,None
8,house_number,VARCHAR,YES,None,None,None
9,healthcare_specialty,VARCHAR,YES,None,None,None


## Row Count

In [3]:
%%sql
SELECT COUNT(*) AS total FROM '{{path}}'

,total
0,1823


## Row Count per Country

In [4]:
%%sql
SELECT country, COUNT(*) AS total
FROM '{{path}}'
GROUP BY country
ORDER BY total DESC

,country,total
0,England,1474
1,Scotland,193
2,Wales,116
3,Northern Ireland,40


## Sample Rows

In [5]:
%%sql
SELECT
    name,
    ST_AsText(geometry) AS geometry_wkt,
    ST_AsText(geometry_3035) AS geometry_3035_wkt,
    country,
    city,
    bronze_path
FROM '{{path}}'
LIMIT 5

,name,geometry_wkt,geometry_3035_wkt,country,city,bronze_path
0,"""Waterside Hospital""",POINT (-7.2804733 55.0135559),POINT (3227375.0541840354 3677318.751521779),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
1,"""Macular Clinic""",POINT (-5.9394996 54.6079957),POINT (3300439.429080259 3613527.498958773),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
2,"""Omagh Hospital & Primary Care Complex""",POINT (-7.2660007 54.5904403),POINT (3216858.485281045 3631198.283217704),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
3,"""Gransha Park Hospital""",POINT (-7.278551 55.015507),POINT (3227546.745480609 3677501.121493304),Northern Ireland,Unknown,s3://bronze/northern_ireland/2026-06-07T09-37-...
4,"""Daisy Hill Hospital""",POINT (-6.3508635 54.1797795),POINT (3263719.991628657 3572846.153332631),Northern Ireland,"""Newry""",s3://bronze/northern_ireland/2026-06-07T09-37-...


## Null Counts

In [6]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE id IS NULL)             AS id_nulls,
    COUNT(*) FILTER (WHERE name IS NULL)           AS name_nulls,
    COUNT(*) FILTER (WHERE geometry IS NULL)       AS geometry_nulls,
    COUNT(*) FILTER (WHERE geometry_3035 IS NULL)  AS geometry_3035_nulls,
    COUNT(*) FILTER (WHERE city IS NULL)           AS city_nulls,
    COUNT(*) FILTER (WHERE postcode IS NULL)       AS postcode_nulls
FROM '{{path}}'

,id_nulls,name_nulls,geometry_nulls,geometry_3035_nulls,city_nulls,postcode_nulls
0,0,0,0,0,0,0


## "Unknown" Counts

In [7]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE name = 'Unknown')           AS name_nulls,
    COUNT(*) FILTER (WHERE city = 'Unknown')           AS city_nulls,
    COUNT(*) FILTER (WHERE postcode = 'Unknown')       AS postcode_nulls,
    COUNT(*) FILTER (WHERE healthcare_specialty = 'Unknown')       AS healthcare_specialty_nulls
FROM '{{path}}'

,name_nulls,city_nulls,postcode_nulls,healthcare_specialty_nulls
0,80,878,812,1644


## Hospitals Within 50km of Lancaster

Uses the `geometry_3035` column (EPSG:3035, metric) so that `ST_Distance` returns metres.

In [8]:
%%sql
WITH lancaster AS (
    SELECT ST_Transform(ST_Point(-2.8013499, 54.0488219), 'EPSG:4326', 'EPSG:3035', true) AS geom
)
SELECT
    name,
    city,
    postcode,
    ROUND(ST_Distance(geometry_3035, lancaster.geom) / 1000.0, 1) AS distance_km,
    ST_AsText(geometry) AS geometry_wkt
FROM '{{path}}', lancaster
WHERE ST_Distance(geometry_3035, lancaster.geom) <= 50000
ORDER BY distance_km

,name,city,postcode,distance_km,geometry_wkt
0,"""The Lancaster Hospital""","""Lancaster""","""LA1 3RH""",0.7,POINT (-2.7952469 54.0431418)
1,"""Royal Lancaster Infirmary""","""Lancaster""","""LA1 4RP""",0.7,POINT (-2.8005027 54.0426284)
2,"""Ashton Community Care Centre""","""Lancaster""","""LA1 4JT""",1.0,POINT (-2.7987449 54.0396994)
3,"""Dacrelands Clinic""","""Lancaster""","""LA1 2DU""",1.2,POINT (-2.8006616 54.0598681)
4,"""DeVitre House""","""Lancaster""","""LA1 5AL""",1.9,POINT (-2.8026259 54.0315277)
5,"""The Orchard""","""Lancaster""","""LA1 4JJ""",1.9,POINT (-2.8040055 54.0321496)
6,"""Queen Victoria Hospital""","""Morecambe""","""LA4 5NN""",4.6,POINT (-2.8588556 54.0727631)
7,"""The Cove""","""Heysham""","""LA3 2SL""",6.1,POINT (-2.8922193 54.0351442)
8,"""Fleetwood Hospital""","""Fleetwood""","""FY7 6BE""",19.3,POINT (-3.0090102 53.9267965)
9,Unknown,Unknown,Unknown,25.6,POINT (-3.0980734 54.1987391)
